# 📅 Dia 2 - Exercício 7: Reação à Cor com Lógica `if/else` 🛑🟢

Agora que já sabemos isolar uma cor na câmara, vamos dar "inteligência" ao JetRacer. Vamos programar um comportamento de reação: **Se** o robô vir um sinal vermelho, ele pára; **Caso contrário** (se o caminho estiver livre), ele avança.

Para isso, vamos contar quantos píxeis brancos existem na nossa imagem filtrada. Se esse número for muito alto, significa que o obstáculo está mesmo à nossa frente!

---

### 🎯 O Teu Objetivo
Escrever ou ajustar uma estrutura de decisão (`if/else`) em Python para fazer o carro andar e parar de forma totalmente automática com base na cor.

### 🛠️ Instruções Passo a Passo

1. **Segurança Máxima:** Confirma que o JetRacer está com as **rodas no ar** (em cima do bloco).
2. **Configura o Alvo:** Usa os sliders `H_min` e `H_max` para calibrar a cor do teu objeto (ex: usa os valores que descobriste no Exercício 6).
3. **Executa a célula** e observa as rodas do robô a girar (Caminho Livre).
4. **Coloca o objeto colorido à frente da câmara:** Vais ver o texto mudar para "🚨 OBSTÁCULO DETETADO!" e as rodas vão **parar de girar** sozinhas!
5. Tira o objeto da frente: As rodas voltam a girar.
6. Quando terminares o teste, clica no botão **❌ DESLIGAR SISTEMA**.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jethelper import mover, parar
import time

print("--- SISTEMA DE DECISÃO (VERSÃO ULTRA-ESTÁVEL) ---")

# 1. Inicializar a câmara
camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15, flip_method=0)

# 2. Criar a Interface Visual
imagem_filtro_widget = widgets.Image(format='jpeg', width=300, height=300)
slider_h_min = widgets.IntSlider(value=0, min=0, max=179, description='H Mínimo:')
slider_h_max = widgets.IntSlider(value=10, min=0, max=179, description='H Máximo:')
botao_desligar = widgets.Button(description="❌ DESLIGAR SISTEMA", button_style='danger')

display(imagem_filtro_widget)
display(widgets.VBox([slider_h_min, slider_h_max, botao_desligar]))

# Variável de controlo para garantir a paragem
sistema_ativo = True

# 3. Função de Processamento e Decisão Otimizada
def analisar_e_tomar_decisao(change):
    global sistema_ativo
    
    # Se o botão de desligar já tiver sido clicado, ignora o processamento e pára o motor
    if not sistema_ativo:
        parar()
        return
        
    frame = change['new']
    
    # Filtro HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    baixo = np.array([slider_h_min.value, 100, 100])
    alto = np.array([slider_h_max.value, 255, 255])
    mascara = cv2.inRange(hsv, baixo, alto)
    
    total_pixeis_cor = np.sum(mascara == 255)
    
    # Lógica de Decisão
    if total_pixeis_cor > 4000:
        print(f"🚨 OBSTÁCULO DETETADO! Píxeis: {total_pixeis_cor} -> TRAVAR!    ", end='\r')
        parar()
    else:
        print(f"🟢 Caminho livre. Píxeis: {total_pixeis_cor} -> Avançar...      ", end='\r')
        mover(0.12, 0.0) # Reduzimos ligeiramente a velocidade para segurança na aula
        
    _, jpeg = cv2.imencode('.jpg', mascara)
    imagem_filtro_widget.value = jpeg.tobytes()
    
    # Dá 20ms de folga ao processador da Jetson para ouvir o clique do rato
    time.sleep(0.02)

# Liga a câmara à função
camera.observe(analisar_e_tomar_decisao, names='value')

# 4. Função do Botão (Força a paragem imediata)
def encerra_sistema(b):
    global sistema_ativo
    sistema_ativo = False # Bloqueia o processamento do if/else acima
    
    print("\n[A parar motores e a desligar câmara...]")
    parar() # Desliga os motores imediatamente
    
    try:
        camera.unobserve(analisar_e_tomar_decisao, names='value')
        camera.running = False
    except:
        pass
        
    botao_desligar.description = "🛑 SISTEMA INATIVO"
    botao_desligar.button_style = "info"
    botao_desligar.disabled = True
    print("Plataforma imobilizada com sucesso!")

botao_desligar.on_click(encerra_sistema)

# 5. Iniciar o sistema
camera.running = True

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


--- SISTEMA DE DECISÃO (VERSÃO ULTRA-ESTÁVEL) ---


Image(value=b'', format='jpeg', height='300', width='300')